In [1]:
import os

print(os.listdir("/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset"))

['WESAD']


In [2]:
import os
import pickle
import warnings
from collections import OrderedDict, defaultdict

import numpy as np
from scipy.signal import butter, filtfilt, find_peaks, welch

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False


# ============================================================
# CHANGE ONLY THIS PATH
# ============================================================
DATA_PATH = "/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD"


# ============================================================
# MAIN SETTINGS
# ============================================================
CHEST_FS = 700
BVP_FS = 64
WINDOW_SEC = 60
BASE_STEP_SEC = 30  # precompute every 30s, then select 30 or 60 during assembly
RANDOM_STATE = 42

# This script aims to IMPROVE LOSO accuracy, not to match the paper table.
# The paper used 10-fold CV, while LOSO is stricter and usually lower.
# To improve LOSO, we search a small configuration space for the best LOSO proxy score.
RUN_CONFIG_SEARCH = True

# Small but useful search space.
SEARCH_LABEL_MODES = ["baseline_only", "baseline_plus_amusement"]
SEARCH_WINDOW_LABEL_MODES = ["pure", "majority"]
SEARCH_STEP_SECONDS = [60, 30]
SEARCH_SENSOR_MODES = ["core", "full"]
SEARCH_EMG_MODES = ["agg", "concat"]
SEARCH_FEATURE_K = [30, 45, "all"]
TOP_SEARCH_RESULTS = 5

# If you want to skip the search later, set RUN_CONFIG_SEARCH = False and put a config here.
MANUAL_CONFIG = {
    "label_mode": "baseline_plus_amusement",
    "window_label_mode": "pure",
    "step_sec": 30,
    "sensor_mode": "full",
    "emg_mode": "agg",
    "feature_k": 45,
}


# ============================================================
# UTILITY FUNCTIONS
# ============================================================
def safe_1d(x):
    x = np.asarray(x, dtype=float).squeeze()
    if x.ndim == 0:
        x = np.asarray([float(x)], dtype=float)
    return x


def subject_sort_key(name):
    try:
        return int(str(name).replace("S", ""))
    except Exception:
        return str(name)


def safe_filtfilt(b, a, x):
    x = safe_1d(x)
    padlen = 3 * (max(len(a), len(b)) - 1)
    if len(x) <= padlen:
        return x
    try:
        return filtfilt(b, a, x)
    except Exception:
        return x


def lowpass(x, cutoff, fs, order=4):
    x = safe_1d(x)
    nyq = 0.5 * fs
    cutoff = min(float(cutoff), nyq * 0.99)
    if cutoff <= 0:
        return x
    try:
        b, a = butter(order, cutoff / nyq, btype="low")
        return safe_filtfilt(b, a, x)
    except Exception:
        return x


def highpass(x, cutoff, fs, order=4):
    x = safe_1d(x)
    nyq = 0.5 * fs
    cutoff = min(float(cutoff), nyq * 0.99)
    if cutoff <= 0:
        return x
    try:
        b, a = butter(order, cutoff / nyq, btype="high")
        return safe_filtfilt(b, a, x)
    except Exception:
        return x


def bandpass(x, low, high, fs, order=4):
    x = safe_1d(x)
    nyq = 0.5 * fs
    low = max(1e-6, min(float(low), nyq * 0.99))
    high = max(1e-6, min(float(high), nyq * 0.99))
    if low >= high:
        return x
    try:
        b, a = butter(order, [low / nyq, high / nyq], btype="band")
        return safe_filtfilt(b, a, x)
    except Exception:
        return x


def robust_mean(x):
    x = safe_1d(x)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else 0.0


def robust_std(x):
    x = safe_1d(x)
    x = x[np.isfinite(x)]
    return float(np.std(x)) if len(x) else 0.0


def robust_median(x):
    x = safe_1d(x)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else 0.0


def robust_min(x):
    x = safe_1d(x)
    x = x[np.isfinite(x)]
    return float(np.min(x)) if len(x) else 0.0


def robust_max(x):
    x = safe_1d(x)
    x = x[np.isfinite(x)]
    return float(np.max(x)) if len(x) else 0.0


def robust_range(x):
    x = safe_1d(x)
    x = x[np.isfinite(x)]
    return float(np.max(x) - np.min(x)) if len(x) else 0.0


def rms(x):
    x = safe_1d(x)
    x = x[np.isfinite(x)]
    return float(np.sqrt(np.mean(x ** 2))) if len(x) else 0.0


def linear_slope(x):
    x = safe_1d(x)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x), dtype=float)
    try:
        p = np.polyfit(t, x, 1)
        return float(p[0])
    except Exception:
        return 0.0


def zero_crossings(x):
    x = safe_1d(x)
    if len(x) < 2:
        return 0.0
    x = x - np.mean(x)
    s = np.sign(x)
    s[s == 0] = 1
    return float(np.sum(s[:-1] != s[1:]))


def slope_sign_changes(x):
    x = safe_1d(x)
    if len(x) < 3:
        return 0.0
    dx = np.diff(x)
    s = np.sign(dx)
    s[s == 0] = 1
    return float(np.sum(s[:-1] != s[1:]))


def band_powers(freqs, psd, bands):
    out = []
    for lo, hi in bands:
        mask = (freqs >= lo) & (freqs < hi)
        if np.any(mask):
            out.append(float(np.trapz(psd[mask], freqs[mask])))
        else:
            out.append(0.0)
    return out


def slice_by_seconds(sig, start_sec, end_sec, fs):
    if sig is None:
        return None
    sig = safe_1d(sig)
    start = int(round(start_sec * fs))
    end = int(round(end_sec * fs))
    start = max(0, min(start, len(sig)))
    end = max(start, min(end, len(sig)))
    return sig[start:end]


def fit_transform_train_test(X_train, y_train, X_test, model_name, feature_k):
    X_train = np.asarray(X_train, dtype=float)
    X_test = np.asarray(X_test, dtype=float)
    y_train = np.asarray(y_train, dtype=int)

    X_train = np.nan_to_num(X_train, nan=np.nan, posinf=np.nan, neginf=np.nan)
    X_test = np.nan_to_num(X_test, nan=np.nan, posinf=np.nan, neginf=np.nan)

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    if model_name in {"Logistic Regression", "LDA", "KNN"}:
        scaler = StandardScaler()
    else:
        scaler = MinMaxScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    selector = None
    if feature_k != "all":
        k = int(min(int(feature_k), X_train.shape[1]))
        if k > 0 and k < X_train.shape[1]:
            selector = SelectKBest(score_func=f_classif, k=k)
            X_train = selector.fit_transform(X_train, y_train)
            X_test = selector.transform(X_test)

    classes, counts = np.unique(y_train, return_counts=True)
    if len(classes) == 2:
        min_count = int(np.min(counts))
        if min_count >= 2:
            k_neighbors = min(5, min_count - 1)
            if k_neighbors >= 1:
                try:
                    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors)
                    X_train, y_train = smote.fit_resample(X_train, y_train)
                except Exception:
                    pass

    return X_train, y_train, X_test


# ============================================================
# FEATURE EXTRACTION
# ============================================================
def heart_features(sig, fs, signal_type="ecg"):
    x = safe_1d(sig)
    if len(x) < fs * 10:
        return [0.0] * 9

    try:
        if signal_type == "ecg":
            x = bandpass(x, 5.0, 15.0, fs, order=2)
            distance = max(1, int(0.35 * fs))
            prominence = max(1e-6, 0.25 * float(np.std(x)))
        else:
            x = bandpass(x, 0.5, 8.0, fs, order=2)
            distance = max(1, int(0.40 * fs))
            prominence = max(1e-6, 0.12 * float(np.std(x)))

        peaks, _ = find_peaks(x, distance=distance, prominence=prominence)
        if len(peaks) < 3:
            return [0.0] * 9

        rr = np.diff(peaks) / float(fs)
        rr = rr[(rr > 0.30) & (rr < 2.0) & np.isfinite(rr)]
        if len(rr) < 2:
            return [0.0] * 9

        hr = 60.0 / rr
        hr = hr[np.isfinite(hr)]
        if len(hr) < 2:
            return [0.0] * 9

        drr = np.diff(rr)
        rmssd = float(np.sqrt(np.mean(drr ** 2))) if len(drr) else 0.0
        pnn50 = float(np.mean(np.abs(drr) > 0.05)) if len(drr) else 0.0

        return [
            float(np.mean(hr)),
            float(np.std(hr)),
            float(np.median(hr)),
            float(np.min(hr)),
            float(np.max(hr)),
            float(np.mean(rr)),
            float(np.std(rr)),
            rmssd,
            pnn50,
        ]
    except Exception:
        return [0.0] * 9


def eda_features(sig, fs):
    x = safe_1d(sig)
    if len(x) < fs * 10:
        return [0.0] * 10

    try:
        x = lowpass(x, 5.0, fs, order=4)
        tonic = lowpass(x, 0.05, fs, order=2)
        phasic = x - tonic

        prominence = max(1e-6, 0.15 * float(np.std(phasic)))
        scr_peaks, _ = find_peaks(
            phasic,
            distance=max(1, int(fs * 1.0)),
            prominence=prominence,
        )

        phasic_auc = float(np.trapz(np.abs(phasic))) / max(1, len(phasic))

        return [
            robust_mean(tonic),
            robust_std(tonic),
            robust_median(tonic),
            robust_range(tonic),
            robust_mean(phasic),
            robust_std(phasic),
            rms(phasic),
            robust_max(phasic),
            phasic_auc,
            float(len(scr_peaks)),
        ]
    except Exception:
        return [0.0] * 10


def resp_features(sig, fs):
    x = safe_1d(sig)
    if len(x) < fs * 10:
        return [0.0] * 8

    try:
        x = bandpass(x, 0.1, 0.35, fs, order=2)
        prominence = max(1e-6, 0.10 * float(np.std(x)))

        maxima, _ = find_peaks(
            x,
            distance=max(1, int(fs * 1.5)),
            prominence=prominence,
        )
        minima, _ = find_peaks(
            -x,
            distance=max(1, int(fs * 1.5)),
            prominence=prominence,
        )

        ibi = np.diff(maxima) / float(fs)
        ibi = ibi[(ibi > 1.0) & (ibi < 10.0) & np.isfinite(ibi)]
        mean_ibi = float(np.mean(ibi)) if len(ibi) else 0.0
        std_ibi = float(np.std(ibi)) if len(ibi) else 0.0
        breath_rate = float(60.0 / mean_ibi) if mean_ibi > 0 else 0.0

        return [
            float(len(maxima)),
            float(len(minima)),
            mean_ibi,
            std_ibi,
            breath_rate,
            robust_mean(np.abs(x)),
            robust_std(x),
            robust_range(x),
        ]
    except Exception:
        return [0.0] * 8


def temp_features(sig):
    x = safe_1d(sig)
    return [
        robust_mean(x),
        robust_std(x),
        robust_min(x),
        robust_max(x),
        robust_range(x),
        linear_slope(x),
    ]


def emg_segment_features(seg, fs):
    seg = safe_1d(seg)
    if len(seg) == 0:
        return [0.0] * 14

    freqs, psd = welch(seg, fs=fs, nperseg=min(1024, len(seg)))
    bands = [
        (0, 50), (50, 100), (100, 150), (150, 200),
        (200, 250), (250, 300), (300, 350)
    ]
    if len(psd) == 0:
        peak_freq = 0.0
        bp = [0.0] * 7
    else:
        peak_freq = float(freqs[np.argmax(psd)])
        bp = band_powers(freqs, psd, bands)

    return [
        float(np.mean(np.abs(seg))),
        float(np.std(seg)),
        rms(seg),
        float(np.sum(np.abs(np.diff(seg)))) if len(seg) > 1 else 0.0,
        zero_crossings(seg),
        slope_sign_changes(seg),
        peak_freq,
    ] + bp


def emg_features_agg(sig, fs):
    x = safe_1d(sig)
    if len(x) < fs * 10:
        return [0.0] * 32

    try:
        hp = highpass(x, 5.0, fs, order=4)
        seg_len = int(fs * 5)
        seg_feats = []

        for start in range(0, len(hp) - seg_len + 1, seg_len):
            seg = hp[start:start + seg_len]
            if len(seg) == seg_len:
                seg_feats.append(emg_segment_features(seg, fs))

        if len(seg_feats) > 0:
            seg_feats = np.asarray(seg_feats, dtype=float)
            part_a = np.mean(seg_feats, axis=0).tolist() + np.std(seg_feats, axis=0).tolist()
        else:
            part_a = [0.0] * 28

        lp = lowpass(x, 50.0, fs, order=4)
        part_b = [
            robust_mean(lp),
            robust_std(lp),
            rms(lp),
            robust_max(np.abs(lp)),
        ]
        return part_a + part_b
    except Exception:
        return [0.0] * 32


def emg_features_concat(sig, fs):
    x = safe_1d(sig)
    expected_len = 12 * 14 + 4
    if len(x) < fs * 10:
        return [0.0] * expected_len

    try:
        hp = highpass(x, 5.0, fs, order=4)
        seg_len = int(fs * 5)
        blocks = []

        for start in range(0, len(hp) - seg_len + 1, seg_len):
            seg = hp[start:start + seg_len]
            if len(seg) == seg_len:
                blocks.extend(emg_segment_features(seg, fs))
            if len(blocks) >= 12 * 14:
                break

        if len(blocks) < 12 * 14:
            blocks.extend([0.0] * (12 * 14 - len(blocks)))

        lp = lowpass(x, 50.0, fs, order=4)
        blocks.extend([
            robust_mean(lp),
            robust_std(lp),
            rms(lp),
            robust_max(np.abs(lp)),
        ])
        return blocks
    except Exception:
        return [0.0] * expected_len


# ============================================================
# DATA LOADING
# ============================================================
def load_subjects(data_path):
    subject_names = sorted(
        [name for name in os.listdir(data_path) if str(name).startswith("S")],
        key=subject_sort_key,
    )

    subjects = []
    for subject in subject_names:
        print(f"Loading {subject} ...")
        with open(os.path.join(data_path, subject, f"{subject}.pkl"), "rb") as f:
            data = pickle.load(f, encoding="latin1")

        chest = data["signal"]["chest"]
        wrist = data["signal"]["wrist"]

        subjects.append({
            "name": subject,
            "labels": safe_1d(data["label"]).astype(int),
            "ECG": safe_1d(chest["ECG"]),
            "EDA": safe_1d(chest["EDA"]),
            "Resp": safe_1d(chest["Resp"]),
            "Temp": safe_1d(chest["Temp"]),
            "EMG": safe_1d(chest["EMG"]),
            "BVP": safe_1d(wrist["BVP"]) if "BVP" in wrist else None,
        })

    return subjects


def summarize_labels(label_block):
    label_block = np.asarray(label_block, dtype=int)
    if len(label_block) == 0:
        return 0, 0, False

    nonzero = label_block[label_block != 0]
    if len(nonzero) == 0:
        return 0, 0, False

    first_nonzero = int(nonzero[0])
    binc = np.bincount(nonzero)
    majority = int(np.argmax(binc)) if len(binc) else 0
    pure = bool(len(np.unique(label_block)) == 1 and label_block[0] != 0)

    return first_nonzero, majority, pure


def precompute_windows(subjects):
    window = int(CHEST_FS * WINDOW_SEC)
    step = int(CHEST_FS * BASE_STEP_SEC)
    all_windows = []

    for subj in subjects:
        usable_len = min(
            len(subj["labels"]),
            len(subj["ECG"]),
            len(subj["EDA"]),
            len(subj["Resp"]),
            len(subj["Temp"]),
            len(subj["EMG"]),
        )

        count = 0
        slot = 0
        for start in range(0, usable_len - window + 1, step):
            end = start + window

            label_block = subj["labels"][start:end]
            first_label, majority_label, pure = summarize_labels(label_block)

            start_sec = start / float(CHEST_FS)
            end_sec = end / float(CHEST_FS)
            bvp_window = slice_by_seconds(subj["BVP"], start_sec, end_sec, BVP_FS)

            ecg = heart_features(subj["ECG"][start:end], CHEST_FS, signal_type="ecg")
            bvp = heart_features(bvp_window, BVP_FS, signal_type="bvp") if bvp_window is not None and len(bvp_window) > 0 else [0.0] * 9
            eda = eda_features(subj["EDA"][start:end], CHEST_FS)
            resp = resp_features(subj["Resp"][start:end], CHEST_FS)
            temp = temp_features(subj["Temp"][start:end])
            emg_agg = emg_features_agg(subj["EMG"][start:end], CHEST_FS)
            emg_concat = emg_features_concat(subj["EMG"][start:end], CHEST_FS)

            hr_diff = [
                abs(ecg[0] - bvp[0]),
                abs(ecg[1] - bvp[1]),
            ]

            all_windows.append({
                "subject": subj["name"],
                "slot": slot,
                "first_label": first_label,
                "majority_label": majority_label,
                "pure": pure,
                "ecg": ecg,
                "bvp": bvp,
                "hr_diff": hr_diff,
                "eda": eda,
                "resp": resp,
                "temp": temp,
                "emg_agg": emg_agg,
                "emg_concat": emg_concat,
            })
            slot += 1
            count += 1

        print(f"Precomputed windows for {subj['name']}: {count}")

    return all_windows


# ============================================================
# DATASET ASSEMBLY
# ============================================================
def map_binary_label(raw_label, label_mode):
    raw_label = int(raw_label)

    if label_mode == "baseline_only":
        if raw_label == 1:
            return 0
        if raw_label == 2:
            return 1
        return None

    if label_mode == "baseline_plus_amusement":
        if raw_label in (1, 3):
            return 0
        if raw_label == 2:
            return 1
        return None

    raise ValueError(f"Unknown label_mode: {label_mode}")


def get_window_label(window_row, window_label_mode):
    if window_label_mode == "pure":
        if not window_row["pure"]:
            return None
        return window_row["first_label"]

    if window_label_mode == "majority":
        return window_row["majority_label"]

    raise ValueError(f"Unknown window_label_mode: {window_label_mode}")


def build_features(window_row, sensor_mode, emg_mode):
    if sensor_mode == "core":
        feats = window_row["ecg"] + window_row["eda"] + window_row["resp"]
    elif sensor_mode == "full":
        emg_block = window_row["emg_agg"] if emg_mode == "agg" else window_row["emg_concat"]
        feats = (
            window_row["ecg"] +
            window_row["bvp"] +
            window_row["hr_diff"] +
            window_row["eda"] +
            window_row["resp"] +
            window_row["temp"] +
            emg_block
        )
    else:
        raise ValueError(f"Unknown sensor_mode: {sensor_mode}")

    feats = np.asarray(feats, dtype=float)
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return feats


def assemble_dataset(all_windows, label_mode, window_label_mode, step_sec, sensor_mode, emg_mode):
    X, y, groups = [], [], []

    for w in all_windows:
        if step_sec == 60 and (w["slot"] % 2 != 0):
            continue

        raw_label = get_window_label(w, window_label_mode)
        if raw_label is None:
            continue

        binary_label = map_binary_label(raw_label, label_mode)
        if binary_label is None:
            continue

        X.append(build_features(w, sensor_mode, emg_mode))
        y.append(binary_label)
        groups.append(w["subject"])

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    groups = np.asarray(groups)
    return X, y, groups


# ============================================================
# MODELS
# ============================================================
def get_models():
    models = OrderedDict()

    models["Decision Tree"] = DecisionTreeClassifier(
        max_depth=5,
        criterion="entropy",
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    models["Logistic Regression"] = LogisticRegression(
        solver="lbfgs",
        max_iter=2500,
        C=2.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    models["LDA"] = LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage="auto",
    )

    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(
            max_depth=4,
            learning_rate=0.05,
            n_estimators=300,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.5,
            min_child_weight=2,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            verbosity=0,
        )

    models["Random Forest"] = RandomForestClassifier(
        n_estimators=500,
        max_depth=14,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    models["KNN"] = KNeighborsClassifier(
        n_neighbors=7,
        weights="distance",
        metric="minkowski",
        p=2,
        n_jobs=-1,
    )

    models["Gradient Boosting"] = GradientBoostingClassifier(
        n_estimators=250,
        learning_rate=0.05,
        subsample=0.9,
        max_depth=3,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
    )

    return models


def get_proxy_model():
    # Fast and usually strong enough to choose a better LOSO config.
    return RandomForestClassifier(
        n_estimators=250,
        max_depth=12,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


# ============================================================
# EVALUATION
# ============================================================
def evaluate_model(model, X, y, groups, model_name, feature_k, verbose=False):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    groups = np.asarray(groups)

    logo = LeaveOneGroupOut()
    accs, precs, recs, f1s = [], [], [], []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), start=1):
        X_train_raw = X[train_idx]
        X_test_raw = X[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]
        test_subject = np.unique(groups[test_idx])[0]

        try:
            X_train, y_train_bal, X_test = fit_transform_train_test(
                X_train_raw,
                y_train,
                X_test_raw,
                model_name,
                feature_k,
            )

            model.fit(X_train, y_train_bal)
            y_pred = model.predict(X_test)
        except Exception as e:
            if verbose:
                print(f"{model_name:20s} | Fold {fold:02d} | Test={test_subject} | ERROR: {e}")
            return None

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        accs.append(acc)
        precs.append(prec)
        recs.append(rec)
        f1s.append(f1)

        if verbose:
            print(
                f"{model_name:20s} | Fold {fold:02d} | Test={test_subject} | "
                f"Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}"
            )

    return {
        "Precision": float(np.mean(precs)),
        "Recall": float(np.mean(recs)),
        "F1-score": float(np.mean(f1s)),
        "Accuracy": float(np.mean(accs)),
    }


# ============================================================
# CONFIG SEARCH FOR BETTER LOSO ACCURACY
# ============================================================
def iter_configs():
    for label_mode in SEARCH_LABEL_MODES:
        for window_label_mode in SEARCH_WINDOW_LABEL_MODES:
            for step_sec in SEARCH_STEP_SECONDS:
                for sensor_mode in SEARCH_SENSOR_MODES:
                    if sensor_mode == "core":
                        for feature_k in SEARCH_FEATURE_K:
                            yield {
                                "label_mode": label_mode,
                                "window_label_mode": window_label_mode,
                                "step_sec": step_sec,
                                "sensor_mode": sensor_mode,
                                "emg_mode": "agg",
                                "feature_k": feature_k,
                            }
                    else:
                        for emg_mode in SEARCH_EMG_MODES:
                            for feature_k in SEARCH_FEATURE_K:
                                yield {
                                    "label_mode": label_mode,
                                    "window_label_mode": window_label_mode,
                                    "step_sec": step_sec,
                                    "sensor_mode": sensor_mode,
                                    "emg_mode": emg_mode,
                                    "feature_k": feature_k,
                                }


def proxy_score(metrics):
    # Slightly favor accuracy, but keep F1 important.
    return 0.60 * metrics["Accuracy"] + 0.40 * metrics["F1-score"]


def search_best_config(all_windows):
    print("\n==============================")
    print("LOSO CONFIG SEARCH (OPTIMIZE ACCURACY, NOT PAPER MATCH)")
    print("==============================")

    rows = []
    proxy = get_proxy_model()

    for cfg in iter_configs():
        X, y, groups = assemble_dataset(
            all_windows=all_windows,
            label_mode=cfg["label_mode"],
            window_label_mode=cfg["window_label_mode"],
            step_sec=cfg["step_sec"],
            sensor_mode=cfg["sensor_mode"],
            emg_mode=cfg["emg_mode"],
        )

        if len(X) == 0 or len(np.unique(y)) < 2 or len(np.unique(groups)) < 2:
            continue

        metrics = evaluate_model(
            model=proxy,
            X=X,
            y=y,
            groups=groups,
            model_name="Random Forest",
            feature_k=cfg["feature_k"],
            verbose=False,
        )
        if metrics is None:
            continue

        score = proxy_score(metrics)
        classes, counts = np.unique(y, return_counts=True)
        class_dist = dict(zip(classes.tolist(), counts.tolist()))

        row = {
            "cfg": cfg,
            "shape": X.shape,
            "class_dist": class_dist,
            "metrics": metrics,
            "score": score,
        }
        rows.append(row)

        print(
            f"labels={cfg['label_mode']:24s} | "
            f"win={cfg['window_label_mode']:8s} | "
            f"step={cfg['step_sec']:2d}s | "
            f"sensors={cfg['sensor_mode']:4s} | "
            f"emg={cfg['emg_mode']:6s} | "
            f"k={str(cfg['feature_k']):3s} | "
            f"X={X.shape} | class={class_dist} | "
            f"Proxy Acc={metrics['Accuracy']:.4f} | Proxy F1={metrics['F1-score']:.4f} | Score={score:.4f}"
        )

    if not rows:
        raise RuntimeError("No valid configuration found in the LOSO search.")

    rows = sorted(rows, key=lambda r: r["score"], reverse=True)

    print("\n==============================")
    print("TOP SEARCH RESULTS")
    print("==============================")
    for i, row in enumerate(rows[:TOP_SEARCH_RESULTS], start=1):
        cfg = row["cfg"]
        m = row["metrics"]
        print(
            f"#{i} | labels={cfg['label_mode']} | win={cfg['window_label_mode']} | "
            f"step={cfg['step_sec']}s | sensors={cfg['sensor_mode']} | emg={cfg['emg_mode']} | "
            f"k={cfg['feature_k']} | X={row['shape']} | class={row['class_dist']} | "
            f"Acc={m['Accuracy']:.4f} | F1={m['F1-score']:.4f} | Score={row['score']:.4f}"
        )

    return rows[0]["cfg"]


# ============================================================
# MAIN
# ============================================================
def main():
    warnings.filterwarnings("ignore")

    print("Loading subjects ...")
    subjects = load_subjects(DATA_PATH)
    print(f"Loaded subjects: {len(subjects)}")

    print("\nPrecomputing windows once ...")
    all_windows = precompute_windows(subjects)
    print(f"Total base windows: {len(all_windows)}")

    if RUN_CONFIG_SEARCH:
        best_cfg = search_best_config(all_windows)
    else:
        best_cfg = MANUAL_CONFIG

    print("\n==============================")
    print("BEST LOSO CONFIG")
    print("==============================")
    for k, v in best_cfg.items():
        print(f"{k:18s}: {v}")

    X, y, groups = assemble_dataset(
        all_windows=all_windows,
        label_mode=best_cfg["label_mode"],
        window_label_mode=best_cfg["window_label_mode"],
        step_sec=best_cfg["step_sec"],
        sensor_mode=best_cfg["sensor_mode"],
        emg_mode=best_cfg["emg_mode"],
    )

    print("\n==============================")
    print("FINAL DATASET USED")
    print("==============================")
    print("Feature matrix:", X.shape)
    unique, counts = np.unique(y, return_counts=True)
    print("Class distribution:", dict(zip(unique.tolist(), counts.tolist())))
    print("Subjects:", len(np.unique(groups)))

    results = OrderedDict()

    print("\n==============================")
    print("FINAL AVERAGE RESULTS (LOSO)")
    print("==============================")
    for model_name, model in get_models().items():
        print(f"\n--- {model_name} ---")
        metrics = evaluate_model(
            model=model,
            X=X,
            y=y,
            groups=groups,
            model_name=model_name,
            feature_k=best_cfg["feature_k"],
            verbose=True,
        )
        if metrics is None:
            print(f"{model_name}: failed")
            continue
        results[model_name] = metrics

    print("\n==============================")
    print("FINAL AVERAGE RESULTS (LOSO)")
    print("==============================")
    for model_name, metrics in results.items():
        print(
            f"{model_name:20s} | "
            f"Precision={metrics['Precision']:.4f} | "
            f"Recall={metrics['Recall']:.4f} | "
            f"F1={metrics['F1-score']:.4f} | "
            f"Accuracy={metrics['Accuracy']:.4f}"
        )

    if results:
        ordered = sorted(results.items(), key=lambda kv: kv[1]["Accuracy"], reverse=True)
        print("\nObserved accuracy order:", " > ".join([k for k, _ in ordered]))

    if not HAS_XGB:
        print("\nXGBoost was skipped because xgboost is not installed.")
        print("Install it with: pip install xgboost")

    if RUN_CONFIG_SEARCH:
        print("\nNOTE:")
        print("This script searched the same dataset to choose a higher-accuracy LOSO configuration.")
        print("That usually improves results, but it also makes the reported numbers slightly optimistic.")
        print("After you find a good config, you can copy it into MANUAL_CONFIG and set RUN_CONFIG_SEARCH=False.")


if __name__ == "__main__":
    main()


Loading subjects ...
Loading S2 ...
Loading S3 ...
Loading S4 ...
Loading S5 ...
Loading S6 ...
Loading S7 ...
Loading S8 ...
Loading S9 ...
Loading S10 ...
Loading S11 ...
Loading S13 ...
Loading S14 ...
Loading S15 ...
Loading S16 ...
Loading S17 ...
Loaded subjects: 15

Precomputing windows once ...
Precomputed windows for S2: 201
Precomputed windows for S3: 215
Precomputed windows for S4: 213
Precomputed windows for S5: 207
Precomputed windows for S6: 234
Precomputed windows for S7: 173
Precomputed windows for S8: 181
Precomputed windows for S9: 173
Precomputed windows for S10: 182
Precomputed windows for S11: 173
Precomputed windows for S13: 183
Precomputed windows for S14: 183
Precomputed windows for S15: 174
Precomputed windows for S16: 186
Precomputed windows for S17: 196
Total base windows: 2874

LOSO CONFIG SEARCH (OPTIMIZE ACCURACY, NOT PAPER MATCH)
labels=baseline_only            | win=pure     | step=60s | sensors=core | emg=agg    | k=30  | X=(434, 27) | class={0: 282, 1: